# Proof of Concept

Calculate a simple forward pass from 3DGS to 3DGS

In [1]:
!pip install gsplat
!pip install plyfile

In [2]:
!git clone https://github.com/facebookresearch/uco3d.git

fatal: destination path 'uco3d' already exists and is not an empty directory.


In [3]:
!pip install -e /content/uco3d/

Obtaining file:///content/uco3d/uco3d
ERROR: file:///content/uco3d/uco3d does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [4]:
import sys

[p for p in sys.path if 'uco3d' in p]

# Remove all uco3d entries from sys.path
sys.path = [p for p in sys.path if p != '/content/uco3d']

# Drop any cached uco3d module
for mod in list(sys.modules):
    if mod == 'uco3d' or mod.startswith('uco3d.'):
        del sys.modules[mod]

# Reimport
import uco3d
print(uco3d.__file__)   # should now be /content/uco3d/uco3d/__init__.py
print(uco3d.__path__)

None
_NamespacePath(['/content/uco3d', '/content/uco3d'])


In [ ]:
!mkdir data/
!python uco3d/dataset_download/download_dataset.py --download_small_subset --download_folder ./data

./data/uco3d_dataset_download_urls.json: 2.01MiB [00:00, 45.5MiB/s]     
  0% 0/1 [00:00<?, ?it/s]Downloading dataset file examples.zip (https://scontent.xx.fbcdn.net/m1/v/t6/An-21J6kqKa3OgaAGbVSYwrzQ8j7KpNsm6BKTuMdLwNltR5fZYioPUD8ldPNVo-D5f3PuA.zip/examples.zip?_nc_gid&ccb=10-5&oh=00_Af4qfwu9YLAW4NIB7wdocNie5fW-1fr1LGBW4_GEqCo5Ug&oe=6A1B3F5F&_nc_sid=636c42) to ./data/_in_progress/examples.zip.
./data/_in_progress/examples.zip:   0% 0.00/8.71G [00:00<?, ?iB/s]examples.zip: Downloaded 0.0%.
./data/_in_progress/examples.zip:   0% 1.00k/8.71G [00:00<33:28, 4.66MiB/s]
./data/_in_progress/examples.zip:   5% 441M/8.71G [00:11<02:09, 68.9MiB/s]examples.zip: Downloaded 5.0%.
./data/_in_progress/examples.zip:   5% 446M/8.71G [00:12<02:08, 68.9MiB/s]
./data/_in_progress/examples.zip:  10% 888M/8.71G [00:19<02:14, 62.6MiB/s]examples.zip: Downloaded 10.0%.
./data/_in_progress/examples.zip:  10% 892M/8.71G [00:19<02:14, 62.6MiB/s]
./data/_in_progress/examples.zip:  15% 1.30G/8.71G [00:27<02:01, 65.

In [ ]:
import os
os.environ["UCO3D_DATASET_ROOT"] = "/content/data"

from uco3d.dataset_utils.utils import get_dataset_root
from uco3d import UCO3DDataset, UCO3DFrameDataBuilder, render_splats, get_all_load_dataset



dataset_root = get_dataset_root(assert_exists=True)
subset_lists_file = os.path.join(
    dataset_root,
    "set_lists",
    "set_lists_all-categories.sqlite",
)

official_dataset = UCO3DDataset(
    subset_lists_file=subset_lists_file,
    subsets=["train"],
    frame_data_builder=UCO3DFrameDataBuilder(
        apply_alignment=True,
        load_images=True,
        load_depths=False,
        load_masks=True,
        load_depth_masks=True,
        load_gaussian_splats=True,
        gaussian_splats_truncate_background=True,
        load_point_clouds=True,
        load_segmented_point_clouds=True,
        load_sparse_point_clouds=True,
        box_crop=True,
        box_crop_context=0.4,
        load_frames_from_videos=True,
        image_height=800,
        image_width=800,
        undistort_loaded_blobs=True,
    )
)

frame_data = official_dataset[0]

In [ ]:
from uco3d.dataset_utils.data_types import GaussianSplats
from torch import nn
import torch

class POC_Network(nn.Module):
    def __init__(self, dim):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, dim)
        )

    def forward(self, x):
        x = self.model(x)
        return x

    def reconstruct_splats(self, splats):
        means = splats.means  # (N, 3)
        sh0 = splats.sh0     # (N, 3)
        scales = splats.scales  # (N, 3)
        quats = splats.quats    # (N, 4)
        opacities = splats.opacities  # (N,)

        flat = torch.cat([means, sh0, scales, quats, opacities.unsqueeze(-1)], dim=-1)

        reconstructed_flat = self.model(flat)
        means_recon = reconstructed_flat[:, 0:3]
        sh0_recon = reconstructed_flat[:, 3:6]
        scales_recon = reconstructed_flat[:, 6:9]
        quats_recon = reconstructed_flat[:, 9:13]
        opacities_recon = reconstructed_flat[:, 13:14].squeeze(-1)

        return GaussianSplats(
            means=means_recon,
            sh0=sh0_recon,
            scales=scales_recon,
            quats=quats_recon,
            opacities=opacities_recon,
            shN=splats.shN,
            fg_mask=splats.fg_mask,
        )

network = POC_Network(14)

In [ ]:
import torch
import torch.nn as nn
import dataclasses
import math
from torch.optim import AdamW
from gsplat import rasterization
from uco3d.dataset_utils.utils import opencv_cameras_projection_from_uco3d
import matplotlib.pyplot as plt
from IPython.display import clear_output

DEVICE     = "cuda"
NUM_EPOCHS = 1000

gt_image = frame_data.image_rgb.unsqueeze(0).to(DEVICE)   # (1, 3, H, W)
fg_mask  = frame_data.fg_probability.unsqueeze(0).to(DEVICE) if frame_data.fg_probability is not None else None
h, w     = gt_image.shape[-2:]
image_size_tensor = torch.tensor([[h, w]], dtype=torch.float32)
Rcv, tvec, camera_matrix = opencv_cameras_projection_from_uco3d(
    frame_data.camera, image_size=image_size_tensor,
)
viewmats = torch.eye(4)[None].to(DEVICE)
viewmats[:, :3, :4] = torch.cat([Rcv, tvec[..., None]], dim=-1)
K = camera_matrix.to(DEVICE)   # (1, 3, 3)

raw_splats = frame_data.sequence_gaussian_splats
input_splats = GaussianSplats(
    means     = raw_splats.means.to(DEVICE).detach(),
    sh0       = raw_splats.sh0.to(DEVICE).detach(),
    scales    = raw_splats.scales.to(DEVICE).detach(),
    quats     = raw_splats.quats.to(DEVICE).detach(),
    opacities = raw_splats.opacities.to(DEVICE).detach(),
    shN       = None,
    fg_mask   = None,
)

def render(means, quats, scales, opacities, sh0):
    colors = sh0.unsqueeze(1)   # (N, 1, 3)
    imgs, _, _ = rasterization(
        means     = means,
        quats     = torch.nn.functional.normalize(quats, p=2, dim=-1),
        scales    = torch.exp(scales),
        opacities = torch.sigmoid(opacities.flatten()),
        colors    = colors,
        viewmats  = viewmats,
        Ks        = K,
        width     = w,
        height    = h,
        sh_degree = 0,
        near_plane= 0.01,
        backgrounds = torch.zeros(1, 3, device=DEVICE),
        packed    = False,
        absgrad   = False,
        sparse_grad = False,
        rasterize_mode = "classic",
        distributed = False,
    )
    return imgs.permute(0, 3, 1, 2)   # (1, 3, H, W)

with torch.no_grad():
    ref_render = render(
        input_splats.means,
        input_splats.quats,
        input_splats.scales,
        input_splats.opacities,
        input_splats.sh0,
    )
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(gt_image[0].permute(1,2,0).cpu()); axes[0].set_title("GT")
    axes[1].imshow(ref_render[0].permute(1,2,0).clamp(0,1).cpu()); axes[1].set_title("Reference render")
    plt.show()

def l2_loss(pred, gt, mask):
    binary_mask = (mask > 0.5).float()
    squared_diff = (pred - gt)**2
    return (squared_diff * binary_mask).sum() / binary_mask.sum().clamp(min=1.0)

network = network.to(DEVICE)
optim   = AdamW(network.parameters())

for epoch in range(NUM_EPOCHS):
    network.train()
    optim.zero_grad()
    recon = network.reconstruct_splats(input_splats)

    rendered = render(
        recon.means,
        recon.quats,
        recon.scales,
        recon.opacities,
        recon.sh0,
    )

    loss = l2_loss(rendered, gt_image, mask=fg_mask)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(network.parameters(), max_norm=1.0)
    optim.step()

    if epoch % 50 == 0:
        network.eval()
        with torch.no_grad():
            eval_rendered = render(
                recon.means.detach(),
                recon.quats.detach(),
                recon.scales.detach(),
                recon.opacities.detach(),
                recon.sh0.detach(),
            )
        clear_output(wait=True)
        print(f"Epoch {epoch:04d}/{NUM_EPOCHS} | Loss={loss.item():.6f}")

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(gt_image[0].permute(1,2,0).cpu())
        axes[0].set_title("Ground Truth")
        axes[1].imshow(ref_render[0].permute(1,2,0).clamp(0,1).cpu())
        axes[1].set_title("Input splats rendered")
        axes[2].imshow(eval_rendered[0].permute(1,2,0).clamp(0,1).cpu())
        axes[2].set_title(f"Network recon (epoch {epoch})")
        plt.tight_layout()
        plt.show()

In [ ]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from gsplat import rasterization
from uco3d.uco3d.dataset_utils.utils import opencv_cameras_projection_from_uco3d
import matplotlib.pyplot as plt
from IPython.display import clear_output

DEVICE = "cuda"
NUM_EPOCHS = 1000
LR = 1e-3

def get_camera_params(frame, device):
    h, w = frame.image_rgb.shape[-2:]
    image_size_tensor = torch.tensor([[h, w]], dtype=torch.float32)

    Rcv, tvec, camera_matrix = opencv_cameras_projection_from_uco3d(
        frame.camera, image_size=image_size_tensor,
    )

    viewmats = torch.eye(4)[None].to(device)
    viewmats[:, :3, :4] = torch.cat([Rcv, tvec[..., None]], dim=-1)

    return viewmats, camera_matrix.to(device), h, w

def render_view(recon_splats, viewmats, Ks, width, height):
    imgs, _, _ = rasterization(
        means       = recon_splats.means,
        quats       = torch.nn.functional.normalize(recon_splats.quats, p=2, dim=-1),
        scales      = torch.exp(recon_splats.scales),
        opacities   = torch.sigmoid(recon_splats.opacities.flatten()),
        colors      = recon_splats.sh0.unsqueeze(1),
        viewmats    = viewmats,
        Ks          = Ks,
        width       = width,
        height      = height,
        sh_degree   = 0,
        near_plane  = 0.01,
        backgrounds = torch.tensor([0.0, 0.0, 0.0], device=DEVICE),
        rasterize_mode = "classic",
    )
    return imgs.permute(0, 3, 1, 2)  # (1, 3, H, W)

def l2_loss(pred, gt, mask):
    binary_mask = (mask > 0.5).float()
    squared_diff = (pred - gt)**2
    return (squared_diff * binary_mask).sum() / binary_mask.sum().clamp(min=1.0)

num_views = 10
training_frames = [official_dataset[i] for i in range(num_views)]

raw_splats = training_frames[0].sequence_gaussian_splats

input_splats = GaussianSplats(
    means     = raw_splats.means.to(DEVICE),
    sh0       = raw_splats.sh0.to(DEVICE),
    scales    = raw_splats.scales.to(DEVICE),
    quats     = raw_splats.quats.to(DEVICE),
    opacities = raw_splats.opacities.to(DEVICE),
    shN       = raw_splats.shN.to(DEVICE) if raw_splats.shN is not None else None,
    fg_mask   = None
)

network = network.to(DEVICE)
optimizer = AdamW(network.parameters(), lr=LR)

for epoch in range(NUM_EPOCHS):
    network.train()
    total_epoch_loss = 0

    for frame_data in training_frames:
        optimizer.zero_grad()

        viewmats, Ks, h, w = get_camera_params(frame_data, DEVICE)
        gt_img = frame_data.image_rgb.unsqueeze(0).to(DEVICE)
        mask = frame_data.fg_probability.unsqueeze(0).to(DEVICE)

        recon = network.reconstruct_splats(input_splats)
        rendered = render_view(recon, viewmats, Ks, w, h)

        loss = l2_loss(rendered, gt_img, mask)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(network.parameters(), max_norm=1.0)
        optimizer.step()

        total_epoch_loss += loss.item()

    if epoch % 20 == 0:
        avg_loss = total_epoch_loss / len(training_frames)
        clear_output(wait=True)
        print(f"Epoch {epoch:04d} | Avg L2 Loss: {avg_loss:.6f}")

        with torch.no_grad():
            binary_mask = (mask > 0.5).float()
            masked_recon = rendered * binary_mask

            fig, axes = plt.subplots(1, 3, figsize=(12, 6))
            axes[0].imshow(gt_img[0].permute(1, 2, 0).cpu().clamp(0, 1))
            axes[0].set_title("GT")
            axes[1].imshow(rendered[0].permute(1, 2, 0).cpu().clamp(0, 1))
            axes[1].set_title(f"Recon (Epoch {epoch})")
            axes[2].imshow(masked_recon[0].permute(1, 2, 0).cpu().clamp(0, 1))
            axes[2].set_title(f"Recon (Foreground Only - Epoch {epoch})")
            plt.show()